# GTSRB-Datensatz Klassifizierung mit YOLO-Modell

Verwenden Sie ein vortrainiertes YOLO-Modell, um die Bilder des GTSRB-Datensatzes zu klassifizieren. Weitere Informationen über das YOLO-Modell bekommen Sie auf diesen Seiten:

- https://docs.ultralytics.com/de/
- https://docs.ultralytics.com/tasks/classify/

## Installation und Importe

Es wird die `ultralytics`-Bibliothek installiert und notwendige Python-Module importiert.

In [6]:
# !pip install ultralytics -q

In [7]:
from sklearn.utils import shuffle
from ultralytics import YOLO
from collections import Counter
import pickle
import os

## Konfiguration der Laufzeitumgebung

Das Skript erkennt automatisch, ob es in Google Colab oder lokal ausgeführt wird, und bindet bei Bedarf Google Drive ein.

Die Beständigkeit der Daten ist wichtig. Da Colab-Laufzeiten temporär sind, ermöglicht die Anbindung von Google Drive den Zugriff auf große Datensätze, ohne sie jedes Mal neu hochladen zu müssen.

In [8]:
# Umgebung erkennen: Colab oder lokal
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    BASE_PATH = '/content/drive/MyDrive/'  # ggf. an Projektordner anpassen
    print("Running in Google Colab – Google Drive wurde eingebunden.")
except ModuleNotFoundError:
    IN_COLAB = False
    BASE_PATH = './'  # lokales Arbeitsverzeichnis, ggf. anpassen
    print("Running locally – Google Drive wird nicht eingebunden.")

Running locally – Google Drive wird nicht eingebunden.


## Datensatz

Je nachdem, ob Sie das Notebook lokal oder in einer Google Colab Umgebung laufen lassen, wird der benötigte Datensatz geladen, werden die Features extrahiert und diese im Anschluss gemischt.

In [9]:
# Pfad zum Validierungsdatensatz – abhängig von der Umgebung ggf. an eigene Ordnerstruktur anpassen
if IN_COLAB:
    valid_dir = '/content/drive/MyDrive/Colab Notebooks/KI-thkoeln/praktikum_2/assets/GTSRB/valid.p'
else:
    valid_dir = os.path.join('dataset', 'valid.p')

with open(valid_dir, mode='rb') as valid_dataset:
    valid = pickle.load(valid_dataset)

X_valid = valid['features']
X_valid = shuffle(X_valid)



## Klassifizierung

In diesem Schritt wird ein vortrainiertes YOLO-Modell (Klassifizierungs-Variante) geladen, um die Verkehrszeichen aus dem GTSRB-Validierungsdatensatz zu verarbeiten.

Da das Standard-YOLO-Modell bereits trainiert wurde, zeigt die Auswertung, welche dieser allgemeinen Klassen am häufigsten mit den Verkehrszeichen assoziiert werden. Am Ende sollen die zehn am häufigsten erkannten Klassen sowie die Gesamtzahl der klassifizierten Bilder ausgegeben werden.

Bei `model = YOLO(...)` kann auch ein feingestimmtes Modell aus Aufgabe 3.2 importiert werden.

In [10]:
model = YOLO("yolo26n-cls.pt")

In [11]:

"""
TODO: Klassifizieren Sie alle 4410 Bilder im Validierungs-Datensatz mit dem YOLO-Modell.
"""

import cv2

top_classes = []

print("Starte Klassifizierung der Validierungsbilder...")
for img in X_valid:
    # YOLO benötigt oft BGR statt RGB
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    # Vorhersage (Stream=True spart Arbeitsspeicher)
    results = model(img_bgr, verbose=False)

    for result in results:
        # Die Klasse mit der höchsten Wahrscheinlichkeit (Top-1) extrahieren
        top1_idx = result.probs.top1
        class_name = result.names[top1_idx]
        top_classes.append(class_name)

print(f"Erfolgreich {len(top_classes)} Bilder klassifiziert.")

Starte Klassifizierung der Validierungsbilder...
Erfolgreich 4410 Bilder klassifiziert.


In [12]:

"""
TODO: Geben Sie die Top 10 Klassen mir Ihrer Häufigkeit aus.

Zum Beispiel:

Top 10 erkannte Klassen:
loupe: 1439
analog_clock: 223
...

"""
# Häufigkeiten zählen
counter = Counter(top_classes)
top10 = counter.most_common(10)

print("\nTop 10 erkannte Klassen:")
print("-" * 30)
for klasse, anzahl in top10:
    print(f"{klasse}: {anzahl}")


Top 10 erkannte Klassen:
------------------------------
loupe: 1227
punching_bag: 279
digital_clock: 243
milk_can: 218
guillotine: 168
hourglass: 129
nematode: 112
birdhouse: 103
pick: 101
analog_clock: 92
